# Fraud Detection with AstroML

This notebook demonstrates two complementary fraud detection approaches:

1. **Synthetic fraud injection** — create labeled benchmark datasets
2. **Deep SVDD** — unsupervised anomaly detection when labels are scarce
3. **GNN-based scoring** — inductive risk scoring on the transaction graph

These mirror the fraud patterns found on the Stellar network:
- Sybil clusters (coordinated fan-out from a controller account)
- Wash trading loops (circular value transfer)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import matplotlib.pyplot as plt

## 1. Inject Synthetic Fraud Patterns

AstroML can inject controlled fraud structures into a clean ledger for benchmarking.

In [ ]:
# Equivalent CLI command:
# python -m astroml.ingestion.synthetic_fraud_injector \
#   --input data/clean_ledger.jsonl \
#   --output data/ledger_with_fraud.jsonl \
#   --sybil-clusters 3 --sybil-cluster-size 8 \
#   --wash-loops 2 --wash-loop-size 5

from astroml.ingestion.synthetic_fraud_injector import SyntheticFraudInjector

injector = SyntheticFraudInjector(
    sybil_clusters=3,
    sybil_cluster_size=8,
    wash_loops=2,
    wash_loop_size=5,
)

# Generate a small synthetic clean ledger for demonstration
clean_ledger = injector.generate_clean_ledger(n_accounts=50, n_transactions=200)
fraudulent_ledger, summary = injector.inject(clean_ledger)

print(f'Clean transactions:     {len(clean_ledger)}')
print(f'After injection:        {len(fraudulent_ledger)}')
print(f'Injected fraud txns:    {summary["injected_count"]}')
print(f'Sybil cluster accounts: {summary["sybil_accounts"]}')
print(f'Wash loop accounts:     {summary["wash_accounts"]}')

## 2. Visualize the Fraud Graph

Plot the transaction graph, highlighting injected fraud nodes.

In [ ]:
import networkx as nx
from astroml.features.transaction_graph import TransactionGraph

tg = TransactionGraph()
for tx in fraudulent_ledger[:100]:  # subset for clarity
    tg.add_transaction(
        from_=tx['source'],
        to=tx['destination'],
        amount=tx['amount'],
        asset=tx.get('asset', 'XLM'),
        timestamp=tx['timestamp'],
    )

G = tg.to_networkx()
fraud_nodes = {tx['source'] for tx in fraudulent_ledger if tx.get('synthetic_fraud')}

node_colors = ['#e53e3e' if n in fraud_nodes else '#4299e1' for n in G.nodes()]

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos, node_color=node_colors, node_size=80,
                 with_labels=False, arrows=True, edge_color='#cbd5e0', width=0.5)
plt.title('Transaction Graph (red = fraud nodes)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. Unsupervised Detection with Deep SVDD

Deep SVDD learns a hypersphere around normal transactions.
Anomalies are points far from the center.

In [ ]:
from astroml.models.deep_svdd_trainer import FraudDetectionDeepSVDD
from astroml.features.node_features import compute_node_features

# Extract features
features = compute_node_features(G)
accounts = list(features.keys())
X = np.array([list(features[a].values()) for a in accounts])
y_true = np.array([1 if a in fraud_nodes else 0 for a in accounts])

print(f'Feature matrix: {X.shape}')
print(f'Fraud accounts: {y_true.sum()} / {len(y_true)}')

# Train Deep SVDD on normal data only
normal_X = X[y_true == 0]
detector = FraudDetectionDeepSVDD(input_dim=X.shape[1], hidden_dim=32, latent_dim=8)
detector.fit(normal_X, epochs=30, lr=1e-3)

# Score all accounts
scores = detector.score_samples(X)
print(f'\nAnomaly scores — normal mean: {scores[y_true==0].mean():.4f}, '
      f'fraud mean: {scores[y_true==1].mean():.4f}')

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report

auc = roc_auc_score(y_true, scores)
print(f'ROC-AUC: {auc:.4f}')

# Threshold at 95th percentile of normal scores
threshold = np.percentile(scores[y_true == 0], 95)
y_pred = (scores > threshold).astype(int)
print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))

## 4. Inductive Risk Scoring with GraphSAGE

For production use, AstroML supports inductive scoring — scoring new accounts
without retraining the model.

In [ ]:
from astroml.pipeline.scoring import InductiveScoringPipeline

pipeline = InductiveScoringPipeline.from_graph(G, feature_dict=features)
pipeline.fit(y_train=y_true, epochs=20)

# Score a new account not seen during training
new_account_features = np.random.randn(X.shape[1])
risk_score = pipeline.score_new_account(new_account_features)
print(f'Risk score for new account: {risk_score:.4f}')

## Summary

| Approach | Labels needed | Strengths |
|---|---|---|
| Deep SVDD | No | Works with unlabeled data, detects novel patterns |
| GraphSAGE | Yes | Higher precision when labels available, inductive |

Next: **`03_transaction_graph_analysis.ipynb`** for temporal analysis.